# Titanic - Modelo Matemático

**Objetivo:** Predecir si un pasajero sobrevivió al hundimiento del Titanic.

**Modelos utilizados:**
- Regresión Logística (modelo principal con ecuación matemática)

In [1]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay, roc_curve, roc_auc_score)
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)

In [3]:
df = pd.read_csv('titanic.csv')
print(f"Dimensiones: {df.shape}")
print(f"Columnas: {df.columns.tolist()}")
df.head(10)

Dimensiones: (891, 12)
Columnas: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


El dataset del Titanic contiene información de los pasajeros a bordo del barco en su viaje inaugural. Cada fila representa un pasajero y las columnas describen sus características:

- **PassengerId:** Identificador único del pasajero.
- **Survived:** Variable objetivo (0 = No sobrevivió, 1 = Sobrevivió).
- **Pclass:** Clase del billete (1 = Primera, 2 = Segunda, 3 = Tercera).
- **Sex:** Sexo del pasajero.
- **Age:** Edad en años.
- **SibSp:** Número de hermanos o cónyuge a bordo.
- **Parch:** Número de padres o hijos a bordo.
- **Fare:** Tarifa pagada por el pasajero.
- **Embarked:** Puerto de embarque (C = Cherburgo, Q = Queenstown, S = Southampton).

## Análisis Exploratorio de Datos

In [4]:
print("Información del Dataset")
df.info()
print()
print("Estadísticas Descriptivas")
df.describe()


Información del Dataset
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB

Estadísticas Descriptivas


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


El conjunto de datos cuenta con 891 registros** y 12 columnas. Las estadísticas descriptivas nos permiten identificar de forma anticipada algunos detalles importantes:

- La **tasa de supervivencia** es de aproximadamente 38%, es decir, solo 342 de 891 pasajeros sobrevivieron.
- La **edad media** es de ~29.7 años, con valores que van desde menos de 1 año hasta 80 años.
- La **tarifa media** es de ~32.2, pero con una desviación estándar alta (~49.7), lo que indica grandes diferencias según la clase.
- La mayoría de los pasajeros viajaban solos (SibSp y Parch con mediana 0).

In [5]:
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(2)
pd.DataFrame({'Valores Nulos': nulos, 'Porcentaje %': pct})

,Valores Nulos,Porcentaje %
PassengerId,0,0.00
Survived,0,0.00
Pclass,0,0.00
Name,0,0.00
Sex,0,0.00
Age,177,19.87
SibSp,0,0.00
Parch,0,0.00
Ticket,0,0.00
Fare,0,0.00


La tabla anterior muestra los valores faltantes por columna. Las columnas más afectadas son:

- **Age:** Aproximadamente el 20% de los valores están ausentes. Se imputa con la mediana para evitar sesgo por valores extremos.
- **Embarked:** Solo 2 registros faltantes; se imputa con la moda (valor más frecuente).
- **Cabin:** Tiene más del 77% de valores nulos, por lo que se descarta del modelo.

Tratar correctamente los valores faltantes es esencial para que los algoritmos de machine learning funcionen correctamente.

In [15]:
sc = df['Survived'].value_counts()
print("Distribución de Supervivencia")
print(f"  No Sobrevivió (0): {sc[0]} pasajeros")
print(f"  Sobrevivió    (1): {sc[1]} pasajeros")
print(f"  Tasa de supervivencia: {sc[1]/len(df)*100:.1f}%")

# Por sexo
print("\nTasa de Supervivencia por Sexo")
ss = df.groupby('Sex')['Survived'].mean() * 100
for sexo, pct in ss.items():
    print(f"  {sexo.capitalize()}: {pct:.1f}%")

# Por clase
print("\nTasa de Supervivencia por Clase")
sc2 = df.groupby('Pclass')['Survived'].mean() * 100
nombres = {1: '1ra Clase', 2: '2da Clase', 3: '3ra Clase'}
for clase, pct in sc2.items():
    print(f"  {nombres[clase]}: {pct:.1f}%")

# Distribución de edades
print("\nDistribución de Edades")
edad = df['Age'].dropna()
print(f"  Mínima:  {edad.min():.0f} años")
print(f"  Máxima:  {edad.max():.0f} años")
print(f"  Promedio: {edad.mean():.1f} años")
print(f"  Mediana:  {edad.median():.1f} años")
print(f"  Datos faltantes en Age: {df['Age'].isnull().sum()}")

Distribución de Supervivencia
  No Sobrevivió (0): 549 pasajeros
  Sobrevivió    (1): 342 pasajeros
  Tasa de supervivencia: 38.4%

Tasa de Supervivencia por Sexo
  Female: 74.2%
  Male: 18.9%

Tasa de Supervivencia por Clase
  1ra Clase: 63.0%
  2da Clase: 47.3%
  3ra Clase: 24.2%

Distribución de Edades
  Mínima:  0 años
  Máxima:  80 años
  Promedio: 29.7 años
  Mediana:  28.0 años
  Datos faltantes en Age: 177


Las gráficas anteriores revelan patrones importantes del dataset:

- **Distribución de supervivencia:** La mayoría de los pasajeros no sobrevivió (~62%). Esto indica un desbalance de clases moderado.
- **Supervivencia por sexo:** Las mujeres sobrevivieron a una tasa mucho mayor que los hombres, reflejando la política de evacuación 'mujeres y niños primero'.
- **Supervivencia por clase:** Los pasajeros de primera clase tuvieron la tasa de supervivencia más alta, seguidos por segunda y tercera clase, lo que sugiere que la posición socioeconómica influyó en el acceso a los botes salvavidas.
- **Distribución de edades:** La mayoría de los pasajeros tenía entre 20 y 40 años, con una distribución ligeramente sesgada hacia la derecha.

## Preprocesamiento de Datos

In [7]:
df_model = df.copy()

df_model['Age']      = df_model['Age'].fillna(df_model['Age'].median())
df_model['Embarked'] = df_model['Embarked'].fillna(df_model['Embarked'].mode()[0])
df_model['Fare']     = df_model['Fare'].fillna(df_model['Fare'].median())

le = LabelEncoder()
df_model['Sex_enc']      = le.fit_transform(df_model['Sex'])       # male=1, female=0
df_model['Embarked_enc'] = le.fit_transform(df_model['Embarked'])  # C=0, Q=1, S=2

features = ['Pclass', 'Sex_enc', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_enc']
X = df_model[features]
y = df_model['Survived']

print("Features seleccionadas:", features)
print(f"Forma X: {X.shape}  |  Forma y: {y.shape}")
print(f"Tasa de supervivencia: {y.mean()*100:.1f}%")
df_model[features + ['Survived']].head()

Features seleccionadas: ['Pclass', 'Sex_enc', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_enc']
Forma X: (891, 7)  |  Forma y: (891,)
Tasa de supervivencia: 38.4%


,Pclass,Sex_enc,Age,SibSp,Parch,Fare,Embarked_enc,Survived
0,3,1,22.0,1,0,7.2500,2,0
1,1,0,38.0,1,0,71.2833,0,1
2,3,0,26.0,0,0,7.9250,2,1
3,1,0,35.0,1,0,53.1000,2,1
4,3,1,35.0,0,0,8.0500,2,0


El preprocesamiento convierte los datos crudos en un formato que los modelos pueden procesar:

1. **Imputación de valores nulos:** Age se rellena con la mediana, Embarked con la moda y Fare con la mediana.
2. **Codificación de variables categóricas:** Se usa LabelEncoder para convertir Sex y Embarked en valores numéricos (female=0, male=1 / C=0, Q=1, S=2).
3. **Selección de características:** Se conservan 7 variables predictoras: Pclass, Sex_enc, Age, SibSp, Parch, Fare y Embarked_enc. Columnas como Name, Ticket y Cabin se descartan por no aportar información útil al modelo.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Entrenamiento: {X_train.shape[0]} muestras")
print(f"Prueba: {X_test.shape[0]} muestras")

Entrenamiento: 712 muestras
Prueba: 179 muestras


El dataset se divide en 80% para entrenamiento y 20% para prueba usando train_test_split. El parámetro stratify=y garantiza que ambas particiones mantengan la misma proporción de supervivientes (~38%), evitando así que el modelo se entrene con datos desbalanceados. El random_state=42 asegura que los resultados sean reproducibles.

## Modelo 1: Regresión Logística

La regresión logística modela la probabilidad de sobrevivir con la función sigmoide:

$$P(Y=1 \mid X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \beta_2 X_2 + \cdots + \beta_n X_n)}}$$

Donde $\beta_i$ son los coeficientes aprendidos durante el entrenamiento.


In [9]:
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)
acc_lr    = accuracy_score(y_test, y_pred_lr)

print(f"Regresion Logistica")
print(f"Exactitud: {acc_lr:.4f}  ({acc_lr*100:.2f}%)")

coef_df = pd.DataFrame({'Feature': features, 'Coeficiente (B)': log_reg.coef_[0].round(4), 'Odds Ratio exp(B)': np.exp(log_reg.coef_[0]).round(4)})
print(f"Intercepto (B0): {log_reg.intercept_[0]:.4f}")
coef_df


Regresion Logistica
Exactitud: 0.8045  (80.45%)
Intercepto (B0): 4.9994


,Feature,Coeficiente (B),Odds Ratio exp(B)
0,Pclass,-1.0467,0.3511
1,Sex_enc,-2.5906,0.0750
2,Age,-0.0380,0.9627
3,SibSp,-0.2472,0.7810
4,Parch,-0.0866,0.9170
5,Fare,0.0022,1.0022
6,Embarked_enc,-0.2305,0.7941


La tabla de coeficientes nos permite interpretar el impacto de cada variable en la predicción:

- El **coeficiente (β)** indica la dirección y magnitud del efecto de cada variable sobre el log-odds de sobrevivir.
- El **Odds Ratio exp(β)** indica cuánto aumenta o disminuye la probabilidad de sobrevivir por cada unidad que aumenta la variable:
  - Un OR > 1 significa que la variable aumenta la probabilidad de sobrevivir.
  - Un OR < 1 significa que la variable reduce dicha probabilidad.
- Por ejemplo, Sex_enc (male=1) con OR < 1 confirma que ser hombre reduce significativamente la probabilidad de supervivencia.

In [10]:
b0 = log_reg.intercept_[0]
b  = log_reg.coef_[0]

print("Ecueacion del modelo")
print("log[ P / (1-P) ] = B0 + B1*Pclass + B2*Sex + B3*Age + ...")
terminos = [f"{b0:.4f}"]
for i, feat in enumerate(features):
    signo = '+' if b[i] >= 0 else '-'
    terminos.append(f"({signo}{abs(b[i]):.4f})*{feat}")
print("log[ P/(1-P) ] = " + " ".join(terminos))
print("=> P(Sobrevive) = 1 / (1 + exp(-Z))   donde Z es la ecuacion anterior")

Ecueacion del modelo
log[ P / (1-P) ] = B0 + B1*Pclass + B2*Sex + B3*Age + ...
log[ P/(1-P) ] = 4.9994 (-1.0467)*Pclass (-2.5906)*Sex_enc (-0.0380)*Age (-0.2472)*SibSp (-0.0866)*Parch (+0.0022)*Fare (-0.2305)*Embarked_enc
=> P(Sobrevive) = 1 / (1 + exp(-Z))   donde Z es la ecuacion anterior


La ecuación anterior es la función logit del modelo, que representa el logaritmo de las probabilidades (log-odds). Para obtener la probabilidad de supervivencia se aplica la función sigmoide:

$$P(\text{Sobrevive}) = \frac{1}{1 + e^{-Z}}$$

donde $Z$ es el valor calculado por la ecuación lineal. Si $P > 0.5$, el modelo predice que el pasajero **sobrevivió**; si $P \leq 0.5$, predice que **no sobrevivió**.

## Modelo 2: Árbol de Decisión

In [11]:
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)
acc_tree    = accuracy_score(y_test, y_pred_tree)

print(f"Exactitud Arbol de Decision: {acc_tree:.4f}  ({acc_tree*100:.2f}%)")

feat_imp = pd.DataFrame({'Caracteristicas': features, 'Importancia': tree.feature_importances_.round(4)}).sort_values('Importancia', ascending=False)
print("Importancia de Variables")
feat_imp


Exactitud Arbol de Decision: 0.7598  (75.98%)
Importancia de Variables


,Caracteristicas,Importancia
1,Sex_enc,0.5307
0,Pclass,0.1804
2,Age,0.1218
5,Fare,0.1141
6,Embarked_enc,0.0343
3,SibSp,0.0112
4,Parch,0.0077


El Árbol de Decisión divide el espacio de características en regiones mediante preguntas binarias (por ejemplo: *¿Es mujer? ¿Tiene más de 30 años?*). La tabla de importancia de variables muestra qué características el árbol utilizó más para tomar decisiones. Se limitó la profundidad a max_depth=5 para evitar sobreajuste (overfitting), donde el modelo memoriza los datos de entrenamiento pero falla al generalizar con datos nuevos.

## Modelo 3: Random Forest

In [12]:
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
acc_rf    = accuracy_score(y_test, y_pred_rf)

print(f"Exactitud Random Forest: {acc_rf:.4f}  ({acc_rf*100:.2f}%)")


Exactitud Random Forest: 0.8101  (81.01%)


El Random Forest es un ensamble de múltiples árboles de decisión entrenados sobre subconjuntos aleatorios de los datos (técnica llamada *bagging*). La predicción final se obtiene por votación mayoritaria entre todos los árboles. Esto lo hace más robusto que un árbol individual, ya que reduce la varianza y el riesgo de sobreajuste. Se utilizaron 100 árboles (n_estimators=100) con profundidad máxima de 5.

## Métricas y Evaluación

In [13]:
print("Classification Report — Regresion Logistica")
print(classification_report(y_test, y_pred_lr, target_names=['No Sobrevivio', 'Sobrevivio']))

comp = pd.DataFrame({'Modelo': ['Regresion Logistica', 'Arbol de Decision', 'Random Forest'], 'Accuracy': [acc_lr, acc_tree, acc_rf]})
comp['Accuracy %'] = (comp['Accuracy'] * 100).round(2).astype(str) + '%'
print("Comparacion de modelos")
comp[['Modelo','Accuracy %']]


Classification Report — Regresion Logistica
               precision    recall  f1-score   support

No Sobrevivio       0.81      0.89      0.85       110
   Sobrevivio       0.79      0.67      0.72        69

     accuracy                           0.80       179
    macro avg       0.80      0.78      0.79       179
 weighted avg       0.80      0.80      0.80       179

Comparacion de modelos


,Modelo,Accuracy %
0,Regresion Logistica,80.45%
1,Arbol de Decision,75.98%
2,Random Forest,81.01%


El **reporte de clasificación** muestra para cada clase:

- **Precision:** De todos los pasajeros que el modelo predijo como sobrevivientes, ¿qué fracción realmente sobrevivió?
- **Recall:** De todos los pasajeros que realmente sobrevivieron, ¿qué fracción identificó el modelo?
- **F1-score:** Media armónica entre precisión y recall; útil cuando hay desbalance de clases.

La tabla comparativa muestra el accuracy (exactitud global) de los tres modelos sobre el conjunto de prueba. Un accuracy más alto indica que el modelo clasifica correctamente una mayor proporción de los pasajeros.

## Predicción con el Modelo

In [14]:
nuevo = pd.DataFrame({
    'Pclass':       [1],
    'Sex_enc':      [0],   
    'Age':          [28],
    'SibSp':        [0],
    'Parch':        [0],
    'Fare':         [100],
    'Embarked_enc': [0] 
})

prob = log_reg.predict_proba(nuevo)[0]
pred = log_reg.predict(nuevo)[0]

print("Pasajero hipotetico")
print("  Sexo: Mujer | Clase: 1ra | Edad: 28 | Tarifa: 100 | Puerto: Cherbourg")
print()
print(f"  P(No Sobrevive) = {prob[0]*100:.2f}%")
print(f"  P(Sobrevive)    = {prob[1]*100:.2f}%")
print()
resultado = "SOBREVIVE" if pred == 1 else "NO SOBREVIVE"
print(f"  Prediccion del modelo: {resultado}")


Pasajero hipotetico
  Sexo: Mujer | Clase: 1ra | Edad: 28 | Tarifa: 100 | Puerto: Cherbourg

  P(No Sobrevive) = 4.27%
  P(Sobrevive)    = 95.73%

  Prediccion del modelo: SOBREVIVE


El modelo aplica la ecuación logística al pasajero hipotético para calcular la probabilidad de supervivencia. En este ejemplo, se define un pasajero con las siguientes características:

- **Sexo:** Mujer (Sex_enc = 0)
- **Clase:** Primera (Pclass = 1)
- **Edad:** 28 años
- **Tarifa:** 100
- **Sin familiares a bordo**
- **Puerto de embarque:** Cherburgo (Embarked_enc = 0)

La probabilidad resultante refleja que una pasajera joven de primera clase tenía altas probabilidades de sobrevivir, lo cual es consistente con los patrones encontrados en el análisis exploratorio.

## Conclusion

### Resultados
| Modelo | Descripción |
|--------|------------|
| **Regresión Logística** | Modelo interpretable con ecuación matemática explícita |
| **Árbol de Decisión** | Reglas visualizables, fácil de interpretar |
| **Random Forest** | Ensamble de árboles, mayor robustez |

### Variables más relevantes
1. **Sex** — Las mujeres tuvieron prioridad de evacuación
2. **Pclass** — Los de 1ra clase tuvieron mayor acceso a botes salvavidas
3. **Age** — Niños pequeños tuvieron prioridad
